In [ ]:
import os
from pathlib import Path

import boto3
from botocore.exceptions import ClientError


def upload_directory(local_dir: Path, bucket: str, prefix: str = ""):
    """Recursively upload a local directory to S3."""
    local_dir = local_dir.resolve()
    if not local_dir.is_dir():
        raise ValueError(f"{local_dir} is not a directory")

    session = boto3.Session()  # uses your default AWS credentials
    s3_client = session.client("s3")

    prefix = prefix.strip("/")

    for root, _, files in os.walk(local_dir):
        for filename in files:
            local_path = Path(root) / filename
            rel_path = local_path.relative_to(local_dir)

            if prefix:
                s3_key = f"{prefix}/{rel_path.as_posix()}"
            else:
                s3_key = rel_path.as_posix()

            print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")
            try:
                s3_client.upload_file(str(local_path), bucket, s3_key)
            except ClientError as e:
                print(f"Failed to upload {local_path}: {e}")